Nota: tempo de execução -> alterar tipo de tempo de execução -> GPU

In [ ]:
############################################################################################################################################################################
# Data Exploration & Preprocessing

#----- Importing libraries ----------------------------------------------------------------------------------------------------------------------------------------------------

import numpy as np # array-processing package
import pandas as pd # data analysis toolkit
import matplotlib.pyplot as plt # static, animated and interactive visualizations
import seaborn as sns # data visualization library
import re # regular expressions
from sklearn.metrics.pairwise import cosine_similarity # computes cosine similarity between samples
from sklearn.preprocessing import StandardScaler # standardizes features by removing the mean and scaling to unit variance
from scipy.sparse import csr_matrix # sparse matrix package for numeric data
from scipy.stats import pearsonr # computes the pearson correlation coefficient
from scipy.sparse.linalg import svds # singular value decomposition for matrix factorization
from sklearn.metrics import mean_squared_error, mean_absolute_error, precision_score, recall_score, f1_score # evaluation metrics
from sklearn.model_selection import train_test_split # splits arrays or matrices into random train and test subsets

In [ ]:
#----- Loading the datasets ----------------------------------------------------------------------------------------------------------------------------------------------------

# Note: the datasets were saved as CSV UTF-8 files; sep or delimiter
#imdb_movies = pd.read_csv('imdb_movies.csv', encoding = 'UTF-8', sep = ';')
#imdb_ratings = pd.read_csv('imdb_ratings.csv', encoding = 'UTF-8', sep = ';')


# Note: in google colab in order to not add the datasets every time, we can use:
#from google.colab import drive
#drive.mount('/content/drive')

path = '/content/drive/MyDrive/projeto/'

imdb_movies = pd.read_csv(path + 'imdb_movies.csv', encoding = 'UTF-8', sep = ';')
imdb_ratings = pd.read_csv(path + 'imdb_ratings.csv', encoding = 'UTF-8', sep = ';')


# Note: .head() doesn't print automatically in the terminal, unless we use e.g. Jupiter Notebook
#print(imdb_movies.head())
#print(imdb_ratings.head())


#imdb_movies.info()
#print('-----------------------------------')
#imdb_ratings.info()



#----- Merging the datasets ----------------------------------------------------------------------------------------------------------------------------------------------------

# Note: inner join returns matching values in both tables
imdb_merged = pd.merge(imdb_movies, imdb_ratings, how = "inner", on = "imdb_title_id")
#print(imdb_merged.head())
#print(imdb_merged.info())


#print(f"Colunas: {list(imdb_merged.columns)}")
#print(len(imdb_merged.columns))

In [ ]:
#----- Missing values ----------------------------------------------------------------------------------------------------------------------------------------------------

missing = imdb_merged.isnull().sum()
#print(missing)

# or

#print('---------------------------------------')
missing_table = pd.DataFrame({'Columns': missing.index, 'Missing values': missing.values})
#print(missing_table)


#----- Dropping columns ----------------------------------------------------------------------------------------------------------------------------------------------------

size = len(imdb_merged)


# Note: inplace = True modifies the original DataFrame
for column in imdb_merged.columns:
    if imdb_merged[column].isnull().sum() > 0.5*size:
        imdb_merged.drop(columns = [column], inplace = True)


# Remove columns that are not useful
imdb_merged.drop(columns = ["production_company", "year"], inplace = True) # date_published has more information and less missing values


#print(f"Colunas restantes: {list(imdb_merged.columns)}")
#print(len(imdb_merged.columns))
#print(imdb_merged.info())

In [ ]:
#----- Handle missing values ----------------------------------------------------------------------------------------------------------------------------------------------------

# Filter columns that still have missing values after dropping some columns
remaining_missing = imdb_merged.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0]

remaining_missing_table = pd.DataFrame({'Columns': remaining_missing.index, 'Missing values': remaining_missing.values})
#print(remaining_missing_table)


# Remove rows where movie titles are invalid
# Note: ^ represents the start of the string, $ represents the end, ^$ means the string is completely empty
imdb_merged = imdb_merged[imdb_merged['original_title'].notnull() & ~imdb_merged['original_title'].str.contains(r'#NAME?|^$', na = False)].copy()


# Remove leading '#' from movie titles if present
# Note: ^# matches a '#' at the beginning of the string
imdb_merged.loc[:, 'original_title'] = imdb_merged['original_title'].str.replace(r'^#', '', regex = True)


# Numeric (float and int) columns with missing values
# Note: .iloc[0] returns the first mode in case there are more than one; fillna() replaces missing values
numeric = imdb_merged.select_dtypes(include = ['number']).columns
imdb_merged[numeric] = imdb_merged[numeric].fillna(imdb_merged[numeric].mode().iloc[0])


# Complement country and language
# Note: agg() applies a function to the group; lambda is an anonymous function; empty checks if the mode is empty
country_mode_per_language = imdb_merged.groupby("language")["country"].agg(lambda x: x.mode()[0] if not x.mode().empty else "NA")
language_mode_per_country = imdb_merged.groupby("country")["language"].agg(lambda x: x.mode()[0] if not x.mode().empty else "NA")


# Fill country with the mode of language
# Note: .map() maps values of series according to input correspondence
imdb_merged.loc[imdb_merged["country"].isna(), "country"] = imdb_merged["language"].map(country_mode_per_language)
# Fill language with the mode of country
imdb_merged.loc[imdb_merged["language"].isna(), "language"] = imdb_merged["country"].map(language_mode_per_country)
# Fill the remaining missing values with NA
imdb_merged.fillna({"country": "NA", "language": "NA"}, inplace = True)


# Handle date_published - Inês
# Check if the date_published column is in the format dd-mm-yyyy
def transform_date_format(x):

    if isinstance(x, str):

        if re.match(r'^\d{4}-\d{2}-\d{2}$', x): # if the date is in the format yyyy-mm-dd
            date = pd.to_datetime(x)
            return date.strftime('%d-%m-%Y')

        elif re.match(r'^\d{4}$', x):  # if the date is only the year
            return f"01-01-{x}"  # return a date with day 01 and month 01: here it can become random if we want, but it is not relevant

    return x # if the date is already in the format dd-mm-yyyy


imdb_merged['date_published'] = imdb_merged['date_published'].apply(transform_date_format)

#print(imdb_merged.info())

In [ ]:
#----- Handle data types - Miguel ----------------------------------------------------------------------------------------------------------------------------------------------------

# Note: optimization of the data types reduces memory usage and improves processing speed while preserving the necessary precision

#for coluna in imdb_merged.select_dtypes(include = ["int64"]).columns:
#    print(f"{coluna}: min = {imdb_merged[coluna].min()}, max = {imdb_merged[coluna].max()}")

#for coluna in imdb_merged.select_dtypes(include = ["float64"]).columns:
#    print(f"{coluna}: min = {imdb_merged[coluna].min()}, max = {imdb_merged[coluna].max()}")

#print(np.iinfo(np.int8))
#print(np.iinfo(np.int16))
#print(np.iinfo(np.int32))

# Floats
for coluna in imdb_merged.select_dtypes(include = ["float64"]).columns:
    imdb_merged[coluna] = imdb_merged[coluna].astype("float32")

    if re.search(r"avg|rating|metascore|mean", coluna):
        imdb_merged[coluna] = imdb_merged[coluna].astype("float16")

    else:
        imdb_merged[coluna] = imdb_merged[coluna].astype("int64")

# Integers
for coluna in imdb_merged.select_dtypes(include = ["int64"]).columns:
    minimo = imdb_merged[coluna].min()
    maximo = imdb_merged[coluna].max()

    if minimo >= -128 and maximo <= 127:
        imdb_merged[coluna] = imdb_merged[coluna].astype("int8")

    elif minimo >= -32768 and maximo <= 32767:
        imdb_merged[coluna] = imdb_merged[coluna].astype("int16")

    elif minimo >= -2147483648 and maximo <= 2147483647:
        imdb_merged[coluna] = imdb_merged[coluna].astype("int32")

#print(imdb_merged.info()) # check the size of the dataset and the changed types

In [ ]:
###################################################################################################################################################################################
# Algorithm Selection & Literature Review

# Note: against what was expected, the colum 'original_title' has more titles in english than 'title'


#----- User-Based Collaborative Filtering ----------------------------------------------------------------------------------------------------------------------------------------------------

# User based filtering- recommend products to a user that similar users have liked.
# For measuring the similarity between two users we can either use pearson correlation or cosine similarity.


# Select relevant columns
movies = imdb_merged[['imdb_title_id', 'original_title', 'avg_vote', 'total_votes']]


# Number of dummy users
# Note: use a small number, otherwise it will take too long to run the code
num_users = 100
user_ids = [f'user_{i}' for i in range(1, num_users + 1)]


# Generate user ratings based on avg_vote and total_votes
# Note: the ratings are generated randomly, but the probability of rating a movie is proportional to the total number of votes
ratings = []

# Note: np.random.rand() generates random numbers between 0 and 1
# Note: np.random.normal() draws random samples from a normal distribution
# Note: np.clip() limits the values in an array
for user in user_ids:
    for _, row in movies.iterrows():
        if np.random.rand() < min(1, row['total_votes'] / 100000): # normalized probability
            # Assign a rating close to the avg_vote with small variation
            rating = np.clip(np.random.normal(row['avg_vote'], 1), 1, 10) # normal distribution around avg_vote
            ratings.append([user, row['imdb_title_id'], round(rating)])


# Dataframe with the dummy ratings
ratings_df = pd.DataFrame(ratings, columns = ['user_id', 'imdb_title_id', 'rating'])
ratings_df.to_csv("dummy_ratings.csv", index = False)
print("Dummy ratings saved to 'dummy_ratings.csv'")


# Pivot the ratings dataframe to create a user-item matrix
ratings_df = pd.read_csv("dummy_ratings.csv")
user_movie_matrix = ratings_df.pivot_table(index = "user_id", columns = "imdb_title_id", values = "rating")
user_movie_matrix = user_movie_matrix.fillna(0) # no rating given = 0


# Choose the similarity method
similarity_method = 'cosine' # or 'pearson'

if similarity_method == 'cosine':
    # Note: cosine_similarity() returns a matrix of shape (n_samples, n_samples) containing the pairwise cosine similarity scores
    user_similarity = cosine_similarity(user_movie_matrix)
    user_sim_df = pd.DataFrame(user_similarity, index = user_movie_matrix.index, columns = user_movie_matrix.index)

elif similarity_method == 'pearson':
    # Note: the result is a square matrix with the same number of rows and columns as the number of users
    user_similarity = user_movie_matrix.T.corr(method = 'pearson') # transpose the matrix to get user-user similarity and not item-item similarity
    user_sim_df = pd.DataFrame(user_similarity, index = user_movie_matrix.index, columns = user_movie_matrix.index)


# Recommend movies based on user similarity
# Note: loc accesses a group of rows and columns by labels
# Note: isin() returns a boolean array, which is used to filter the dataframe
def recommend_movies(user_id, n = 20):
    if user_id not in user_sim_df.index:
        return f"User {user_id} not found."

    # Show the movies rated by the user to understand if the recommendations make sense
    user_rated_movies = user_movie_matrix.loc[user_id][user_movie_matrix.loc[user_id] > 0]
    rated_movies_info = imdb_merged[imdb_merged['imdb_title_id'].isin(user_rated_movies.index)][['original_title', 'description', 'avg_vote']]
    rated_movies_info = rated_movies_info.sort_values(by = 'avg_vote', ascending = False)

    print(f"Movies rated by {user_id}:\n")
    print(rated_movies_info)
    print("\n------------------------------\n")

    # Find similar users, excluding the user itself
    similar_users = user_sim_df[user_id].drop(user_id).sort_values(ascending = False)
    top_user = similar_users.index[0] # most similar user - top similar user

    # Recommend movies rated by top_user that the target user hasn't rated
    user_movies = set(user_movie_matrix.loc[user_id][user_movie_matrix.loc[user_id] > 0].index)
    top_user_movies = set(user_movie_matrix.loc[top_user][user_movie_matrix.loc[top_user] > 0].index)
    recommendations = list(top_user_movies - user_movies)

    # Return top n recommendations
    return imdb_merged[imdb_merged['imdb_title_id'].isin(recommendations)][['original_title', 'description', 'avg_vote']].head(n)


# Tryout
print(recommend_movies('user_1'))

Dummy ratings saved to 'dummy_ratings.csv'
Movies rated by user_1:

                     original_title  \
28453      The Shawshank Redemption   
15528                 The Godfather   
16556        The Godfather: Part II   
48078               The Dark Knight   
38407  Hababam Sinifi Sinifta Kaldi   
...                             ...   
55494                Disaster Movie   
39348  Superbabies: Baby Geniuses 2   
85745         Adventures of Aladdin   
51757      Emret Komutanim: Sah Mat   
60133                  Killer Bitch   

                                             description  avg_vote  
28453  Two imprisoned men bond over a number of years...  9.296875  
15528  The aging patriarch of an organized crime dyna...  9.203125  
16556  The early life and career of Vito Corleone in ...  9.000000  
48078  When the menace known as the Joker wreaks havo...  9.000000  
38407  A young and beautiful female teacher starts wo...  9.000000  
...                                              

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


In [ ]:
# ----- Item-Based Collaborative Filtering ------------------------------------------------------------------------------------------------------------------

# Item Based Collaborative Filtering - recommends items based on similarity with the items that the target user rated.
# The similarity can be computed with Pearson Correlation or Cosine Similarity.


from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np


# Choose similarity method
sim_method = 'cosine' # or 'pearson'


# Load ratings
ratings_df = pd.read_csv("dummy_ratings.csv")


# Select relevant columns
movies = imdb_merged[['imdb_title_id', 'original_title', 'avg_vote', 'total_votes']]


# User-item matrix
user_movie_matrix = ratings_df.pivot(index = 'user_id', columns = 'imdb_title_id', values = 'rating')
user_movie_matrix = user_movie_matrix.fillna(0)


# Item-item similarity matrix
if sim_method == 'cosine':
    item_similarity = cosine_similarity(user_movie_matrix.T)
    item_similarity_df = pd.DataFrame(item_similarity, index = user_movie_matrix.columns, columns = user_movie_matrix.columns)

elif sim_method == 'pearson':
    item_similarity_df = user_movie_matrix.corr(method = 'pearson')

else:
    raise ValueError("Invalid similarity method. Use 'cosine' or 'pearson'.")


# Get movie info
def get_movie_info(movie_id):
    row = imdb_merged[imdb_merged['imdb_title_id'] == movie_id]
    if not row.empty:
        return row[['original_title', 'description', 'avg_vote']].values[0]
    else:
        return ["Title not found", "Unknown description", 0]


# Recommend similar movies
def recommend_item(movie_id, num_rec = 10):
    if movie_id not in item_similarity_df.index:
        return []

    similares = item_similarity_df[movie_id].sort_values(ascending = False)[1 : num_rec+1].index
    recommendation = []

    for rec_id in similares:
        title, description, avg_vote = get_movie_info(rec_id)
        recommendation.append({'imdb_title_id': rec_id, 'original_title': title, 'description': description, 'avg_vote': avg_vote})

    return pd.DataFrame(recommendation)


# Tryout
user_id = ratings_df['user_id'].sample(1, random_state = 3112).values[0]
rated_movies_ids = ratings_df[ratings_df['user_id'] == user_id]['imdb_title_id'].tolist()


# Choose a random rated movie for testing
movie_test = np.random.choice(rated_movies_ids)
title, description, avg_vote = get_movie_info(movie_test)

print(f"Movie rated by {user_id} selected:\n")
print(f"Title: {title}")
print(f"Description: {description}")
print(f"Average vote: {avg_vote}")
print("\n" + "-"*30 + "\n")


print(f"Recommended movies based on '{title}' (method: {sim_method}):\n")
recs_df = recommend_item(movie_test, num_rec = 10)
print(recs_df[['original_title', 'description', 'avg_vote']])

Movie rated by user_7 selected for testing:

Title: Rise of the Guardians
Description: When the evil spirit Pitch launches an assault on Earth, the Immortal Guardians team up to protect the innocence of children all around the world.
Average vote: 7.30078125

------------------------------

Recommended movies based on 'Rise of the Guardians' (method: cosine):

  original_title                                        description  avg_vote
0   12 Angry Men  A jury holdout attempts to prevent a miscarria...  8.898438
1   Molly's Game  The true story of Molly Bloom, an Olympic-clas...  7.398438
2     Robin Hood  The story of the legendary outlaw is portrayed...  7.601562
3  Planet Terror  After an experimental bio-weapon is released, ...  7.101562
4  Trainspotting  Renton, deeply immersed in the Edinburgh drug ...  8.101562


In [ ]:
#----- Model-Based Collaborative - Matrix Factorization ------------------------------------------------------------------------------------------------------------------------------

# Matrix Factorization - decomposes the user-item matrix into two lower-dimensional matrices, one for users and one for items.


# Ensure consistency between user-movie matrix and movie metadata
valid_ids = imdb_merged['imdb_title_id'].unique()
filtered_matrix = user_movie_matrix.loc[:, user_movie_matrix.columns.isin(valid_ids)]


# Ensure movie metadata only contains relevant movies
relevant_movies = imdb_merged[imdb_merged['imdb_title_id'].isin(filtered_matrix.columns)].copy()
relevant_movies.drop_duplicates(subset = 'imdb_title_id', inplace = True)
relevant_movies.set_index('imdb_title_id', inplace = True)


# Note: k controls the dimensionality of the latent space and the level of compression
k = min(50, min(filtered_matrix.shape)-1) # number of latent factors


# SVD
# Note: U is the user matrix, S is the singular values (importance of each factor), Vt is the item matrix
def perform_svd(user_movie_matrix, k):
    U, S, Vt = svds(user_movie_matrix, k = k)
    S = np.diag(S) # convert singular values into a diagonal matrix to allow matrix multiplication
    return U, S, Vt

U, S, Vt = perform_svd(filtered_matrix.values, k)


# Predicted ratings matrix
predicted_ratings = np.dot(np.dot(U, S), Vt)
predicted_ratings_df = pd.DataFrame(predicted_ratings, index = filtered_matrix.index, columns = filtered_matrix.columns)


def recommend_movies_svd(user_id, n = 10):
    if user_id not in predicted_ratings_df.index:
        return f"User {user_id} not found."

    # Get the movies with highest predicted ratings for the user
    user_ratings = predicted_ratings_df.loc[user_id].sort_values(ascending = False)

    # Filter out movies already rated by the user
    already_rated = filtered_matrix.loc[user_id][filtered_matrix.loc[user_id] > 0].index
    recommended_movies = user_ratings.drop(index = already_rated, errors = 'ignore').head(n)

    # Filter only valid movies
    valid_recs = recommended_movies.index.intersection(relevant_movies.index)

    recommended_df = relevant_movies.loc[valid_recs][['original_title', 'description', 'avg_vote']]
    recommended_df = recommended_df.sort_values(by = 'avg_vote', ascending = False)

    return recommended_df


# Tryout
user_example = filtered_matrix.sample(n = 1, random_state = 3112).index[0] # random user


# Movies previously rated by the user
rated_ids = filtered_matrix.loc[user_example][filtered_matrix.loc[user_example] > 0].index
rated_ids = rated_ids.intersection(relevant_movies.index)

user_movies = relevant_movies.loc[rated_ids][['original_title', 'description', 'avg_vote']]

print(f"Recommendation for: {user_example}\n")
print("Movies previously rated by the user:")
print(user_movies.sort_values(by = 'avg_vote', ascending = False)) # from most liked to least liked
print("\n--------------------\n")
print("Recommended movies:")
print(recommend_movies_svd(user_example, n = 10))

Recommendation for: user_6

Movies previously rated by the user:
                         original_title  \
imdb_title_id                             
tt0111161      The Shawshank Redemption   
tt0068646                 The Godfather   
tt0249795                   Maya Bazaar   
tt5354160                      Aynabaji   
tt0468569               The Dark Knight   
...                                 ...   
tt1213644                Disaster Movie   
tt0804492       The Hottie & the Nottie   
tt1158700                    Desh Drohi   
tt5988370                          Reis   
tt4404474                Kartoffelsalat   

                                                     description  avg_vote  
imdb_title_id                                                               
tt0111161      Two imprisoned men bond over a number of years...  9.296875  
tt0068646      The aging patriarch of an organized crime dyna...  9.203125  
tt0249795      Balarama promises Subhadra to get his daughter...  9

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


In [ ]:
####################################################################################################################################################################################
# Cold Start Problem

# Note: The cold start problem occurs when a recommender system cannot make accurate recommendations due to the lack of data about new users or new items.


# ----- New Users --------------------------------------------------------------------------------------------------------------------------------------------------------------------

# Cosine similarity between users
cos_sim = cosine_similarity(user_movie_matrix)


# Recommend the most popular items
def recommend_popular_items(user_movie_matrix, n_recommendations = 5):
    item_popularity = user_movie_matrix.astype(bool).sum(axis = 0) # count how many users rated each item
    popular_items = item_popularity.sort_values(ascending = False).head(n_recommendations).index.tolist() # top n most popular items
    return popular_items


# Recommend popular items for a new user
def recommend_for_new_user(user_movie_matrix, n_recommendations = 5):
    return recommend_popular_items(user_movie_matrix, n_recommendations)


# Recommend similar items for an existing user based on the items they have rated
# Note: if user has no ratings or does not exist, return empty list. This way we can check if the picked user already existed.
def recommend_similar_items_based_on_popularity(user_id, user_movie_matrix, n_recommendations = 5):
    if user_id not in user_movie_matrix.index:
        return [] # cold start detected

    user_ratings = user_movie_matrix.loc[user_id]
    rated_items = user_ratings[user_ratings > 0].index

    if len(rated_items) == 0:
        return [] # user has no rated items, can't compute similarity

    candidate_items = user_ratings[user_ratings == 0].index # items the user has not rated

    similar_items = []

    # For each item rated by the user, compute its similarity to unrated items
    for item in rated_items:
        item_vector = user_movie_matrix[item].values.reshape(1, -1) # rated item
        candidate_vectors = user_movie_matrix[candidate_items].values.T # candidate items
        similarities = cosine_similarity(item_vector, candidate_vectors).flatten() # cosine similarity

        similar_items.extend(zip(candidate_items, similarities)) # store the candidate items and their similarity scores

    similar_items = sorted(similar_items, key = lambda x: x[1], reverse = True)

    # Top n unique recommended items
    recommended = []
    seen = set()

    # Iterate over sorted similar items and add unique ones to the recommendation list
    for item, _ in similar_items:
        if item not in seen:
            recommended.append(item)
            seen.add(item)
        if len(recommended) == n_recommendations:
            break

    return recommended


# Tryout
new_user_id = 100 # this user does not exist (cold start scenario)
popular_recommendations = recommend_for_new_user(user_movie_matrix)
similar_item_recommendations = recommend_similar_items_based_on_popularity(new_user_id, user_movie_matrix)


# Convert movies IDs to titles
popular_titles = imdb_merged[imdb_merged['imdb_title_id'].isin(popular_recommendations)]['original_title'].tolist()
similar_titles = imdb_merged[imdb_merged['imdb_title_id'].isin(similar_item_recommendations)]['original_title'].tolist()


print(f"Popular recommendations for new user {new_user_id}: {popular_titles}")
print(f"Similarity-based recommendations for new user {new_user_id}: {similar_titles}")

Popular recommendations for new user 100: ['The Kid', 'Alien³', 'Basic Instinct', 'Batman Returns', 'El Camino: A Breaking Bad Movie']
Similarity-based recommendations for new user 100: []


In [ ]:
# ----- New Movies --------------------------------------------------------------------------------------------------------------------------------------------------------------------


from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd


# Combine textual features into a single string
def combine_features(row):
    return ' '.join([str(row['genre']), str(row['description']), str(row['director']), str(row['actors']), str(row['writer'])])


# Fill missing values with empty strings
imdb_merged[['genre', 'description', 'director', 'actors', 'writer']] = imdb_merged[['genre', 'description', 'director', 'actors', 'writer']].fillna('')


# Create a new column for the textual features
imdb_merged['combined_features'] = imdb_merged.apply(combine_features, axis = 1)


# TF-IDF (term frequency-inverse document frequency)
# Note: F-IDF measures the importance of string representations such as words, phrases and more
# Note: we remove common stop words (e.g., the, and, or of) to reduce noise
tfidf = TfidfVectorizer(stop_words = 'english') # converts a collection of raw documents to a matrix of TF-IDF features
tfidf_matrix = tfidf.fit_transform(imdb_merged['combined_features']) # (n_movies, n_features)


# Recommend similar movies to a new movie
def recommend_similar_movies_for_new_item(new_movie_data, n_recommendations = 5):
    # Note: new_movie_data is a dict with the same keys used in combine_features
    new_movie_text = ' '.join([str(new_movie_data.get('genre', '')), str(new_movie_data.get('description', '')), str(new_movie_data.get('director', '')), str(new_movie_data.get('actors', '')), str(new_movie_data.get('writer', ''))])

    # Transform new movie text
    new_tfidf = tfidf.transform([new_movie_text]) # transforms text into a sparse matrix of n-gram counts

    # Cosine similarity between new movie and all existing ones
    similarity_scores = cosine_similarity(new_tfidf, tfidf_matrix).flatten()

    # Get indices of top N most similar movies
    top_indices = similarity_scores.argsort()[::-1][:n_recommendations]

    return imdb_merged.iloc[top_indices][['imdb_title_id', 'title', 'genre', 'description']]


# Tryout
new_movie_info = {'genre': 'Action Crime', 'description': 'A secret agent embarks on a mission to stop an international crime syndicate.', 'director': 'Rob Cohen', 'actors': 'Tom Cruise, Vin Diesel', 'writer': 'Bruce Geller'}

recommendations = recommend_similar_movies_for_new_item(new_movie_info)

print("Recommendations for the new movie:")
print(recommendations)

Recommendations for the new movie:
      imdb_title_id                          title  \
33108     tt0149171                         Strays   
40863     tt0295701                            xXx   
37418     tt0232500               Fast and Furious   
76126     tt4912910  Mission: Impossible - Fallout   
31291     tt0120755         Mission: Impossible II   

                             genre  \
33108                 Crime, Drama   
40863  Action, Adventure, Thriller   
37418      Action, Crime, Thriller   
76126  Action, Adventure, Thriller   
31291  Action, Adventure, Thriller   

                                             description  
33108  A macho cruiser comes of age. Frustrated by th...  
40863  An extreme sports athlete, Xander Cage, is rec...  
37418  Los Angeles police officer Brian O'Conner must...  
76126  Ethan Hunt and his IMF team, along with some f...  
31291  IMF agent Ethan Hunt is sent to Sydney to find...  


In [ ]:
####################################################################################################################################################################################
# Evaluation Metrics

from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from scipy.sparse.linalg import svds
import numpy as np
import pandas as pd


# Load data
ratings_df = pd.read_csv("dummy_ratings.csv")
ratings_df['rating'] = pd.to_numeric(ratings_df['rating'], errors = 'coerce') # errors = 'coerce: invalid parsing will be set as NaN
ratings_df = ratings_df.dropna(subset = ['rating'])

ratings_df['imdb_title_id'] = ratings_df['imdb_title_id'].astype(str)
imdb_merged['imdb_title_id'] = imdb_merged['imdb_title_id'].astype(str)


# Split train/test per user
def split_train_test(ratings_df, test_size = 0.2):

    train_list = []
    test_list = []

    for user in ratings_df['user_id'].unique():
        user_ratings = ratings_df[ratings_df['user_id'] == user]

        if len(user_ratings) >= 5:
            train, test = train_test_split(user_ratings, test_size = test_size, random_state = 3112)
            train_list.append(train)
            test_list.append(test)
        else:
            train_list.append(user_ratings)

    train_df = pd.concat(train_list)
    test_df = pd.concat(test_list)
    return train_df, test_df

train_df, test_df = split_train_test(ratings_df)


# Build train user-item matrix
user_movie_matrix = train_df.pivot(index = 'user_id', columns = 'imdb_title_id', values = 'rating').fillna(0)


# RMSE and MAE
def evaluate_rating_predictions(actual_matrix, predicted_matrix):
    mask = actual_matrix > 0
    actual = actual_matrix[mask].to_numpy().flatten()
    predicted = predicted_matrix[mask].to_numpy().flatten()

    valid = (~np.isnan(actual)) & (~np.isnan(predicted)) & (~np.isinf(actual)) & (~np.isinf(predicted))
    actual = actual[valid]
    predicted = predicted[valid]

    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mae = mean_absolute_error(actual, predicted)
    return round(rmse, 4), round(mae, 4)


# Top-N Evaluation using test set
def evaluate_top_n(user_id, recommend_func, test_df, n = 10, threshold = 5.0):
    print(f"\nEvaluating Top-N for user: {user_id}")

    user_test_data = test_df[test_df['user_id'] == user_id]
    relevant_items = user_test_data[user_test_data['rating'] >= threshold]['imdb_title_id'].tolist()

    print(f"Relevant items: {relevant_items}")

    recommended_df = recommend_func(user_id, n = n)

    if recommended_df.empty:
        print("Recommended DataFrame is empty.")
    else:
        print(f"Recommended items: {recommended_df['imdb_title_id'].tolist()}")

    recommended_items = set(recommended_df['imdb_title_id']) if not recommended_df.empty else set()
    relevant_items = set(relevant_items)

    if not recommended_items:
        print("No recommendations returned.")
        return {"Precision@N": 0, "Recall@N": 0, "F1-score": 0}

    true_positives = len(recommended_items & relevant_items)
    precision = true_positives / len(recommended_items)
    recall = true_positives / len(relevant_items) if relevant_items else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

    print(f"True Positives: {true_positives}, Precision: {precision}, Recall: {recall}, F1: {f1}")

    return {"Precision@N": round(precision, 4), "Recall@N": round(recall, 4), "F1-score": round(f1, 4)}


# Models and recommenders
def generate_model(method, user_movie_matrix, imdb_merged, similarity = 'cosine'):


    if method == "user_based":

        if similarity == 'cosine':
            sim = cosine_similarity(user_movie_matrix)
        elif similarity == 'pearson':
            sim = user_movie_matrix.T.corr(method = 'pearson').fillna(0).values

        sim_df = pd.DataFrame(sim, index = user_movie_matrix.index, columns = user_movie_matrix.index)

        def recommend(user_id, n = 10):

            if user_id not in sim_df.index:
                return pd.DataFrame()

            similar_users = sim_df.loc[user_id].drop(user_id).sort_values(ascending = False)
            top_user = similar_users.idxmax()
            user_seen = set(user_movie_matrix.loc[user_id][user_movie_matrix.loc[user_id] > 0].index)
            top_user_seen = set(user_movie_matrix.loc[top_user][user_movie_matrix.loc[top_user] > 0].index)
            recommendations = list(top_user_seen - user_seen)

            return imdb_merged[imdb_merged['imdb_title_id'].isin(recommendations)][['imdb_title_id', 'original_title']].head(n)

        weights = sim_df.div(sim_df.sum(axis = 1), axis = 0).fillna(0)
        predicted = weights.dot(user_movie_matrix)

        return predicted, recommend


    elif method == "item_based":

        item_matrix = user_movie_matrix.T

        if similarity == 'cosine':
            sim = cosine_similarity(item_matrix)
        elif similarity == 'pearson':
            sim = item_matrix.corr(method = 'pearson').fillna(0)

        sim_df = pd.DataFrame(sim, index = item_matrix.index, columns = item_matrix.index)
        weights = sim_df.div(sim_df.sum(axis = 1), axis = 0).fillna(0)
        predicted = weights.dot(item_matrix).T

        def recommend(user_id, n = 10):
            if user_id not in predicted.index:
                return pd.DataFrame()

            user_ratings = predicted.loc[user_id].sort_values(ascending = False)
            already_rated = user_movie_matrix.loc[user_id][user_movie_matrix.loc[user_id] > 0].index
            to_recommend = user_ratings.drop(index = already_rated).head(n).index

            return imdb_merged[imdb_merged['imdb_title_id'].isin(to_recommend)][['imdb_title_id', 'original_title']]

        return predicted, recommend


    elif method == "matrix_factorization":

        k = min(50, min(user_movie_matrix.shape) - 1)
        U, S, Vt = svds(user_movie_matrix.values, k = k)
        S_diag = np.diag(S)

        predicted = np.dot(np.dot(U, S_diag), Vt)
        predicted_df = pd.DataFrame(predicted, index = user_movie_matrix.index, columns = user_movie_matrix.columns)

        def recommend(user_id, n = 10):
            if user_id not in predicted_df.index:
                return pd.DataFrame()

            user_ratings = predicted_df.loc[user_id].sort_values(ascending = False)
            already_rated = user_movie_matrix.loc[user_id][user_movie_matrix.loc[user_id] > 0].index
            to_recommend = user_ratings.drop(index = already_rated).head(n).index

            return imdb_merged[imdb_merged['imdb_title_id'].isin(to_recommend)][['imdb_title_id', 'original_title']]

        return predicted_df, recommend


# Model evaluation
models_to_compare = [("user_based", "cosine"), ("user_based", "pearson"), ("item_based", "cosine"), ("item_based", "pearson"), ("matrix_factorization", None)]

eligible_users = test_df[test_df['rating'] >= 5.0]['user_id'].unique()
example_user = np.random.choice(eligible_users)

results = []

print(f"Example user: {example_user}")

for method, sim in models_to_compare:
    print(f"\nEvaluating method: {method} | Similarity: {sim}")

    model_output, recommender = generate_model(method, user_movie_matrix, imdb_merged, sim) if sim else generate_model(method, user_movie_matrix, imdb_merged)
    model_output = model_output.fillna(0)

    rmse, mae = evaluate_rating_predictions(user_movie_matrix, model_output)
    topn_metrics = evaluate_top_n(example_user, recommender, test_df, n = 10)

    print(f"RMSE: {rmse}, MAE: {mae}")
    print(f"Top-N Metrics: {topn_metrics}")

    results.append({"method": method, "similarity": sim, "RMSE": rmse, "MAE": mae, **topn_metrics})


# Summary
pd.set_option("display.max_columns", None)
summary_df = pd.DataFrame(results)

print("\nEvaluation Summary:\n")
print(summary_df)

Example user: user_3

Evaluating method: user_based | Similarity: cosine

Evaluating Top-N for user: user_3
Relevant items: ['tt1051904', 'tt0337921', 'tt0114709', 'tt0815244', 'tt0048281', 'tt0108052', 'tt3901826', 'tt2631186', 'tt2343793', 'tt0103704', 'tt0048728', 'tt0473705', 'tt0499549', 'tt5323662', 'tt0034514', 'tt3011894', 'tt0177858', 'tt0053291', 'tt0119174', 'tt1504320', 'tt1371111', 'tt6751668', 'tt2752772', 'tt0035446', 'tt0775489', 'tt1444262', 'tt1528854', 'tt0455944', 'tt1080016', 'tt0166896', 'tt0090859', 'tt0403702', 'tt9484998', 'tt1456661', 'tt0106341', 'tt5610554', 'tt0116213', 'tt2347569', 'tt0082089', 'tt0059742', 'tt0259324', 'tt5186714', 'tt1053424', 'tt0023926', 'tt1441953', 'tt0113416', 'tt0277371', 'tt1605783', 'tt0109830', 'tt0105698', 'tt0452608', 'tt0097770', 'tt1904996', 'tt1540741', 'tt0444682', 'tt0059749', 'tt6292852', 'tt0109836', 'tt5065790', 'tt2452386', 'tt0263488', 'tt0368619', 'tt0192255', 'tt4303340', 'tt0308644', 'tt0092843', 'tt0083866', 'tt2

In [ ]:
##########################################################################################################################################################################
# Parameter Tuning and Evaluation

import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse.linalg import svds


# Parameters
param_grid_user_based = {'similarity': ['cosine', 'pearson'], 'n_neighbors': [5, 10, 15]}
param_grid_item_based = {'similarity': ['cosine', 'pearson'], 'n_neighbors': [5, 10, 15]}
param_grid_matrix_factorization = {'k': [30, 40, 50, 60]}


def evaluate_predictions(actual_matrix, predicted_matrix):
    # Ensure both matrices are float
    actual_matrix = actual_matrix.astype(float)
    predicted_matrix = predicted_matrix.astype(float)

    # Create mask where actual ratings exist (non-NaN)
    mask = actual_matrix.notna()

    # Actual and predicted values
    actual = actual_matrix[mask].values.flatten()
    predicted = predicted_matrix[mask].values.flatten()

    # RMSE and MAE
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mae = mean_absolute_error(actual, predicted)
    return round(rmse, 4), round(mae, 4)


# User-Based Collaborative Filtering
def evaluate_user_based_model(user_movie_matrix, similarity_metric = 'cosine', n_neighbors = 5):
    matrix_filled = user_movie_matrix.fillna(0).astype(float)

    if similarity_metric == 'cosine':
        sim_matrix = cosine_similarity(matrix_filled)
    elif similarity_metric == 'pearson':
        sim_matrix = matrix_filled.T.corr(method = 'pearson').fillna(0).values

    sim_df = pd.DataFrame(sim_matrix, index = user_movie_matrix.index, columns = user_movie_matrix.index)

    # Top-k neighbors
    top_k_similarities = pd.DataFrame(0.0, index = sim_df.index, columns = sim_df.columns)
    for i in sim_df.index:
        top_k = sim_df.loc[i].nlargest(n_neighbors + 1).drop(i, errors = 'ignore') # +1 includes self, errors = 'ignore': invalid parsing will return the input
        top_k_similarities.loc[i, top_k.index] = top_k.values

    # Normalize weights
    weights = top_k_similarities.div(top_k_similarities.sum(axis = 1), axis = 0).fillna(0)

    # Predict ratings
    predicted_ratings = weights.dot(matrix_filled)

    return pd.DataFrame(predicted_ratings, index = user_movie_matrix.index, columns = user_movie_matrix.columns)


# Item-Based Collaborative Filtering
def evaluate_item_based_model(user_movie_matrix, similarity_metric = 'cosine', n_neighbors = 5):
    matrix_filled = user_movie_matrix.fillna(0).astype(float)

    if similarity_metric == 'cosine':
        sim_matrix = cosine_similarity(matrix_filled.T)
    elif similarity_metric == 'pearson':
        sim_matrix = matrix_filled.corr(method='pearson').fillna(0).values

    sim_df = pd.DataFrame(sim_matrix, index = user_movie_matrix.columns, columns = user_movie_matrix.columns)

    # Top-k neighbors
    top_k_weights = pd.DataFrame(0.0, index = sim_df.index, columns = sim_df.columns)
    for i in sim_df.index:
        top_k = sim_df.loc[i].nlargest(n_neighbors + 1).drop(i, errors = 'ignore')
        top_k_weights.loc[i, top_k.index] = top_k.values

    # Normalize weights
    top_k_weights = top_k_weights.div(top_k_weights.sum(axis = 1), axis = 0).fillna(0)

    # Predict ratings
    predicted_ratings = top_k_weights.dot(matrix_filled.T).T

    return pd.DataFrame(predicted_ratings, index = user_movie_matrix.index, columns = user_movie_matrix.columns)


# Matrix Factorization (SVD)
def evaluate_matrix_factorization(user_movie_matrix, k = 20):
    matrix_filled = user_movie_matrix.fillna(0).astype(float)

    U, S, Vt = svds(matrix_filled.values, k = k)
    S_diag = np.diag(S)
    predicted_ratings = np.dot(np.dot(U, S_diag), Vt)

    predicted_df = pd.DataFrame(predicted_ratings, index = user_movie_matrix.index, columns = user_movie_matrix.columns)

    return predicted_df.fillna(0)


# Evaluation Execution
print("Manual Tuning - User-Based Collaborative Filtering")
for sim in param_grid_user_based['similarity']:

    for k in param_grid_user_based['n_neighbors']:
        predicted = evaluate_user_based_model(user_movie_matrix, similarity_metric = sim, n_neighbors = k)
        rmse, mae = evaluate_predictions(user_movie_matrix, predicted)

        print(f"Similarity: {sim}, Neighbors: {k} --> RMSE: {rmse}, MAE: {mae}")


print("\nManual Tuning - Item-Based Collaborative Filtering")
for sim in param_grid_item_based['similarity']:

    for k in param_grid_item_based['n_neighbors']:
        predicted = evaluate_item_based_model(user_movie_matrix, similarity_metric = sim, n_neighbors = k)
        rmse, mae = evaluate_predictions(user_movie_matrix, predicted)

        print(f"Similarity: {sim}, Neighbors: {k} --> RMSE: {rmse}, MAE: {mae}")


print("\nManual Tuning - Matrix Factorization (SVD)")
for k in param_grid_matrix_factorization['k']:
    predicted = evaluate_matrix_factorization(user_movie_matrix, k = k)
    rmse, mae = evaluate_predictions(user_movie_matrix, predicted)

    print(f"Latent factors (k): {k} --> RMSE: {rmse}, MAE: {mae}")

Manual Tuning - User-Based Collaborative Filtering
Similarity: cosine, Neighbors: 5 --> RMSE: 1.6664, MAE: 0.6829
Similarity: cosine, Neighbors: 10 --> RMSE: 1.5975, MAE: 0.6784
Similarity: cosine, Neighbors: 20 --> RMSE: 1.5641, MAE: 0.6775
Similarity: pearson, Neighbors: 5 --> RMSE: 1.6663, MAE: 0.6827
Similarity: pearson, Neighbors: 10 --> RMSE: 1.5976, MAE: 0.6782
Similarity: pearson, Neighbors: 20 --> RMSE: 1.5642, MAE: 0.6774

Manual Tuning - Item-Based Collaborative Filtering
Similarity: cosine, Neighbors: 5 --> RMSE: 1.4912, MAE: 0.519
Similarity: cosine, Neighbors: 10 --> RMSE: 1.4811, MAE: 0.5313
Similarity: cosine, Neighbors: 20 --> RMSE: 1.5063, MAE: 0.5519
Similarity: pearson, Neighbors: 5 --> RMSE: 1.2294, MAE: 0.3839
Similarity: pearson, Neighbors: 10 --> RMSE: 1.2142, MAE: 0.3926
Similarity: pearson, Neighbors: 20 --> RMSE: 1.2375, MAE: 0.4091

Manual Tuning - Matrix Factorization (SVD)
Latent factors (k): 10 --> RMSE: 1.4515, MAE: 0.6958
Latent factors (k): 20 --> RMSE

In [ ]:
### TAREFAS #######################################################################################################################################################################

### fazer hybrid recommender systems
### fazer e limpar código geral
### trabalhar no relatório
